#


Для аналитики в первую очередь нужны данные:)

В этом разделе мы посмотрим как:

- создать набор пространственных данных на основе координат
- экспортировать данные из OpenStreetMap
- загрузить имеющиеся данные в проект


## Импортируем библиотеки


In [ ]:
import pandas as pd
import geopandas as gpd
import osmnx as ox

<p style="color:#26528C; font-style:italic"> 
<strong>Pandas</strong> пригодится для обработки и анализа структурированных данных (табличек)
</p>

<p style="color:#26528C; font-style:italic;"> 
<strong>Geopandas</strong> расширяет функциональность Pandas для работы с пространственными данными 
</p>

<p style="color:#26528C; font-style:italic;"> 
<strong>OSMnx</strong> помогает выгружать данные из OpenStreetMap
</p>


## 1.1 Создаем


В первых двух разделах мы представим, что данных у нас совсем нет, и нужно начинать с чистого листа

Знаем только координаты интересующего нас объекта: 59°55′54″ с. ш. 30°20′11″ в. д. Именно его мы хотим отобразить на карте


### Пространственные данные из координат


Долготу и широту присваиваем переменным lat и lon, соответственно, а переменной point присваиваем словарь, в который записываем координты и название объекта


<p style="color:#26528C; font-style:italic"><strong>Словарь</strong> — это тип данных, представляющий собой коллекцию пар «ключ-значение» </p>


In [ ]:
lat =  59.9316667
lon =  30.3363889

point = {'Longitude': lon, 'Latitude': lat, "name": 'Alexandrinsky theater'}

print(point)

На основе словаря (point) создаем DataFrame (point_df). Проверяем получившийся набор данных, чтобы убедиться, что получилось что-то похожее на таблицу


<p style="color:#26528C; font-style:italic"><strong>DataFrame</strong> - это структура данных в библиотеке Pandas, которая представляет собой двумерный массив с индексами для строк и столбцов (aka табличка)</p>


In [ ]:
point_df = pd.DataFrame(point, [1])

point_df

На основе DataFrame (point_df) создаем набор пространственных данных - GeoDataFrame с помощью библиотеки geopandas. Для этого указываем наш массив данных (point_df), определяем геометрию на основе координат и не забываем про систему координат!

Мы используем CRS 4326 - WGS84 - географическую систему координат, так как у нас широта и долгота в градусах.


<p style="color:#26528C; font-style:italic"><strong>GeoDataFrame</strong> - это расширенный DataFrame, у которого есть поле с геометрией объекта </p>


In [ ]:
point_gdf = gpd.GeoDataFrame(point_df, geometry=gpd.points_from_xy(point_df['Longitude'], point_df['Latitude']), crs=4326)

Смотрим на получившийся GeoDataFrame (point_gdf) и убеждаемся, что на одно поле стало больше - добавилась геометрия


In [ ]:
point_gdf

Согласитесь, что по результатам выше не совсем понятно, получили мы новый массив с дополнительным полем или это уже принципиально иной формат данных

Тип данных проверим, используя функцию type()


In [ ]:
print(type(point_df) == type(point_gdf))

print("Тип point_df: %s, Тип point_gdf: %s" % (type(point_df), type(point_gdf)))

<p style="color:#26528C; font-style:italic">Загадочная запись во второй строке внутри print() – форматирование строки с помощью оператора %. В ситуации, когда нужно подставить в строку значения из переменных это очень удобно</p>


Действительно, в двух переменных содержатся данные разного типа, и point_gdf - это GeoDataFrame. Значит, мы успешно создали набор пространственных данных и можем ~~собой гордиться~~ отобразить точку на карте

Для этого воспользуемся методом explore. Он использует библиотеки folium, mapclassify и matplotlib (важно, чтобы они были установлены, дополнительно импортировать в проект их не нужно)


In [ ]:
point_gdf.explore()

Вуаля! Мы видим на интерактивной карте точку, которую создали! Мы не забыли про систему координат и указали ее верно, поэтому Александринский театр оказался там, где и должен быть - в центре Санкт-Петербурга <3


### Сохраняем результат


Теперь мы можем сохранить полученные данные в любой из известных нам форматов с помощью метода to_file


In [ ]:
#point_gdf.to_file('data/dataframe.shp') 

<p style="color:#26528C; font-style:italic">Код в строке выше закомментирован с помощью символа <strong>#</strong>, он может быть выполнен только если убрать комментирование</p>


## 1.2 Экспортируем из OSM


В ооооочень большом числе проектов используются данные OpenStreetMap. Есть множество способов, как их оттуда экспортировать.

Так как мы собираемся анализировать пространственные данные на Python, то и необходимые данные мы можем получить, не закрывая Jupyter Notebook. Для этого воспользуемся библиотекой OSMnx


Для начала стоит определить территорию, на которую мы хотим получить данные. Пусть это будет Центральный район Санкт-Петербурга. Воспользуемся функцией geocode_to_gdf, чтобы получить границы района.


In [ ]:
admin_district = ox.geocode_to_gdf('Центральный район, Санкт-Петербург')

Посмотрим, что мы получили (в методе explore зададим параметр tiles и поменяем базовую карту OSM на Positron)


In [ ]:
admin_district.explore(tiles='cartodbpositron')

Невероятно! Но это именно то, что нам нужно – границы Центрального района Санкт-Петербурга


Теперь попробуем выгрузить здания для Центрального района. Для этого создадим переменную tags, куда передадим ключ нужных нам объектов - buildings.
В функцию geometries_from_place передадим название района и теги объектов

Подробнее о ключах и значениях данных OSM можно прочитать <a href="https://wiki.openstreetmap.org/wiki/Map_features" target="_blank">тут</a>


In [ ]:
tags = {'building': True}   

buildings = ox.features_from_place('Центральный район, Санкт-Петербург', tags)  #in the OSMnx versions before 1.7.0 use ox.geometries.geometries_from_place())

buildings.explore(tiles='cartodbpositron',tooltip=None)

Мы получили здания!

Теперь выберем только полигональные и мультиполигональные объекты


In [ ]:
buildings = buildings.loc[buildings.geom_type.isin(['Polygon', 'MultiPolygon'])]

И смело можем сохранять!

Спойлер – при сохранении возникнет ошибка, так как некоторые поля содержат данные в формате списка (list). Это не подходит для сохранения, поэтому мы это исправим: перезапишем поля как строки(string)


In [ ]:
for col in buildings.columns:
    if any(isinstance(val, list) for val in buildings[col]):
        buildings = buildings.astype({col: str})

И теперь точно можем сохранять!


In [ ]:
#buildings.to_file('data/spb_central_build.geojson')

## 1.3 Читаем


Часто мы работаем с данными, которые уже есть у нас. Давайте попробуем прочитать их. Для пространственных данных будем использовать функцию read_file из библиотеки geopandas, а для csv - read_csv из библиотеки pandas


### Форматы пространственных данных (топ-3)


<p style="color:#26528C; font-style:italic"> 
<strong>Shapefile (SHP)</strong>: стандартный формат для хранения географических векторных данных от Esri. Включает файлы .shp, .shx, .dbf и другие.
</p>
<p style="color:#26528C; font-style:italic"> 
<strong>GeoJSON</strong>: формат, предназначенный для хранения пространственных данных и основан на формате JSON (по сути является его разновидностью).
</p>
<p style="color:#26528C; font-style:italic"> 
<strong>Geopackage (GPKG)</strong>: файл базы данных, который может включать множество наборов данных различных типов (растровые, векторные, табличные, и даже проекты). Все содержится в одном файле с расширением ".gpkg".
</p>


### GeoJSON (или Shapefile)


In [ ]:
metro = gpd.read_file('data/spb_metro.geojson')

In [ ]:
metro.explore()

Готово!)


### GeoPackage


In [ ]:
admin = gpd.read_file('data/spb_admin.gpkg')

Готово! но...давайте взглянем на первые строки атрибутивной таблицы...


In [ ]:
admin.head()

Если у нас в geoPackage содержался только один слой, то все хорошо. Но geoPackage может включать больше наборов данных

Узнать, какие слои содержит geoPackage поможет функция listlayers из библиотеки fiona. Импортируем библиотеку и проверим


In [ ]:
import fiona

layers = fiona.listlayers('data/spb_admin.gpkg')

print(layers)

Теперь мы знаем, какие слои есть в нашем geoPackage, и можем обращаться к ним по имени


In [ ]:
admin_okrug = gpd.read_file('data/spb_admin.gpkg', layer="okrug")

In [ ]:
admin_okrug.explore()

Готово!)


### CSV


<p style="color:#26528C; font-style:italic"> 
<strong>CSV(Comma-Separated Values)</strong> - текстовый формат, используемый для хранения табличных данных. 
Это не формат пространственных данных, но мы в работе с ними часто встречаемся. И иногда это таблицы с объектами, где в одном (или нескольких) полях содержатся координаты.
</p>


In [ ]:
theaters = pd.read_csv('data/spb_theaters.csv')

Посмотрим на верхнюю часть нашей таблицы...


In [ ]:
theaters.head()

Это список театров в Санкт-Петербурге. Их координаты содержатся в столбце coord

Разделим координаты на два столбца (долгота и широта), удалим исходный и посмотрим на результат


In [ ]:
theaters[['latitude', 'longitude']] = theaters['coord'].str.split(',', expand=True)

theaters.drop(columns=["coord"], inplace = True)

theaters.head()

Создадим GeoDataFrame


In [ ]:
theaters_gdf = gpd.GeoDataFrame(theaters, geometry=gpd.points_from_xy(theaters['longitude'], theaters['latitude']), crs=4326)

Удалим строки с пустой геометрией (если такие есть) и посмотрим на результат


In [ ]:
theaters_gdf = theaters_gdf[~theaters_gdf.geometry.is_empty]

theaters_gdf.explore(tiles='cartodbpositron')

## 1.4 Исследуем данные


После того как вы загрузили пространственный набор данных в `GeoDataFrame`, важно понять, что он содержит.
Вот некоторые ключевые характеристики, которые можно изучить, чтобы лучше разобраться в своих данных.


#### Основная информация


In [ ]:
theaters_gdf.info()

Отображает сводную информацию о DataFrame: количество записей, названия столбцов, типы данных, объем используемой памяти


#### Предпросмотр данных


In [ ]:
theaters_gdf.head()

Показывает первые 5 строк набора данных — быстрый способ понять его структуру и содержимое.


#### Количество объектов


In [ ]:
print(len(theaters_gdf))
# or
print(theaters_gdf.shape)

Ввозвращает число строк (объектов). Атрибут `.shape` также показывает количество столбцов.


#### Тип геометрии


In [ ]:
theaters_gdf.geom_type.unique()

Сообщает, какие типы геометрий есть в наборе данных (например, Point, Polygon).


#### Система координат


In [ ]:
theaters_gdf.crs

Показывает систему координат — например, EPSG:4326 (WGS84).


#### Ограничивающий прямоугольник (Bounding Box)


In [ ]:
theaters_gdf.total_bounds

Возвращает границы набора данных: `[minx, miny, maxx, maxy]`.


#### Поле с геометрией


In [ ]:
theaters_gdf.geometry

Возвращает геометрию всех объектов


#### Атрибутивные поля


In [ ]:
theaters_gdf.columns

Перечисляет все столбцы (поля) в GeoDataFrame


#### Обзор


In [ ]:
print("CRS:", theaters_gdf.crs)
print("Number of features:", len(theaters_gdf))
print("Geometry types:", theaters_gdf.geom_type.unique())
print("Bounds:", theaters_gdf.total_bounds)
theaters_gdf.head()

Это позволяет быстро получить общее представление о том, что содержат ваши пространственные данные и как они структурированы — важный шаг перед анализом или визуализацией.


## Итоги


Мы узнали, как создавать данные на основе координат, как экспортировать данные из OSM и как читать данные разных форматов (а еще немного и про сами форматы)
